In [4]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("data/yearly")

yearly_files = {
    year: DATA_DIR / f"citysense_{year}.parquet"
    for year in range(2020, 2026)
}

for year, path in yearly_files.items():
    print(year, path, path.exists())

2020 data\yearly\citysense_2020.parquet True
2021 data\yearly\citysense_2021.parquet True
2022 data\yearly\citysense_2022.parquet True
2023 data\yearly\citysense_2023.parquet True
2024 data\yearly\citysense_2024.parquet True
2025 data\yearly\citysense_2025.parquet True


In [5]:
for year, path in yearly_files.items():
    df_sample = pd.read_parquet(path)
    print(f"{year}: {df_sample.shape}")
    print(df_sample.dtypes)
    print()

2020: (8309664, 16)
grid_id                                          category
cmplnt_fr_dt                               datetime64[us]
hour                                                 int8
crime_count                                         int32
historical_grid_crime_count                         int32
historical_grid_hour_crime_count                    int32
day_of_week                                          int8
historical_grid_day_crime_count                     int32
time_period                                      category
historical_grid_time_period_crime_count             int32
is_weekend                                           int8
historical_grid_weekend_crime_count                 int32
year                                                int64
month                                                int8
lat_grid                                          float32
lon_grid                                          float32
dtype: object

2021: (8286960, 16)
grid_id          

In [6]:
for year in range(2020, 2026):
    df = pd.read_parquet(
        yearly_files[year],
        columns=["crime_count"]
    )
    
    print(
        year,
        "rows:", len(df),
        "crime rate:", (df["crime_count"] > 0).mean()
    )
    
    del df

2020 rows: 8309664 crime rate: 0.04407915891665415
2021 rows: 8286960 crime rate: 0.04764931892998156
2022 rows: 8286960 crime rate: 0.05537434716711556
2023 rows: 8286960 crime rate: 0.05738232114068367
2024 rows: 8309664 crime rate: 0.05864292467180382
2025 rows: 8286960 crime rate: 0.058033464623939296


In [7]:
for year in range(2020, 2024):
    df = pd.read_parquet(
        yearly_files[year],
        columns=["crime_count"]
    )

    print(f"\n{year}")
    print(df["crime_count"].value_counts().sort_index().head(10))
    
    del df


2020
crime_count
0    7943381
1     329739
2      31787
3       3814
4        610
5        153
6         81
7         29
8         17
9         16
Name: count, dtype: int64

2021
crime_count
0    7892092
1     352761
2      36591
3       4545
4        720
5        160
6         39
7         22
8          7
9          6
Name: count, dtype: int64

2022
crime_count
0    7828075
1     404124
2      46658
3       6656
4       1113
5        215
6         71
7         23
8         10
9          6
Name: count, dtype: int64

2023
crime_count
0    7811435
1     416447
2      50147
3       7321
4       1266
5        242
6         60
7         18
8         12
9          2
Name: count, dtype: int64


In [8]:
train_distribution = {}

for year in range(2020, 2024):
    df = pd.read_parquet(
        yearly_files[year],
        columns=["crime_count"]
    )
    
    train_distribution[year] = df["crime_count"].value_counts()
    del df

combined_distribution = sum(train_distribution.values())

print(combined_distribution.sort_index().head(15))
print()
print("Total training rows:", combined_distribution.sum())
print(
    "Overall positive rate:",
    1 - combined_distribution.get(0, 0) / combined_distribution.sum()
)

crime_count
0     31474983.0
1      1503071.0
2       165183.0
3        22336.0
4         3709.0
5          770.0
6          251.0
7           92.0
8           46.0
9           30.0
10          22.0
11          14.0
12           NaN
13           NaN
14           NaN
Name: count, dtype: float64

Total training rows: 33170513.0
Overall positive rate: 0.05111557967161984


In [9]:
import numpy as np

TOTAL_SAMPLE = 1_000_000
train_years = range(2020, 2024)

year_rows = {}

for year in train_years:
    year_rows[year] = len(
        pd.read_parquet(
            yearly_files[year],
            columns=["crime_count"]
        )
    )

total_train_rows = sum(year_rows.values())

sample_sizes = {
    year: round(TOTAL_SAMPLE * rows / total_train_rows)
    for year, rows in year_rows.items()
}

difference = TOTAL_SAMPLE - sum(sample_sizes.values())
sample_sizes[2023] += difference

print("Training rows by year:")
for year in train_years:
    print(year, sample_sizes[year])

print("Total:", sum(sample_sizes.values()))

Training rows by year:
2020 250513
2021 249829
2022 249829
2023 249829
Total: 1000000


In [10]:
import pyarrow.parquet as pq

In [11]:
train_samples = []

for year in train_years:
    df = pd.read_parquet(yearly_files[year])

    n_sample = sample_sizes[year]

    zero_rows = df[df["crime_count"] == 0]
    positive_rows = df[df["crime_count"] > 0]

    positive_fraction = len(positive_rows) / len(df)
    n_positive = round(n_sample * positive_fraction)
    n_zero = n_sample - n_positive

    zero_sample = zero_rows.sample(
        n=n_zero,
        random_state=42
    )

    positive_sample = positive_rows.sample(
        n=n_positive,
        random_state=42
    )

    year_sample = pd.concat(
        [zero_sample, positive_sample],
        ignore_index=True
    )

    train_samples.append(year_sample)

    print(
        year,
        "sample:", len(year_sample),
        "positive:", n_positive,
        "zero:", n_zero
    )

    del df, zero_rows, positive_rows

train_sample = pd.concat(
    train_samples,
    ignore_index=True
)

del train_samples

train_sample = train_sample.sort_values(
    ["year", "cmplnt_fr_dt", "hour", "grid_id"]
).reset_index(drop=True)

print("\nFinal shape:", train_sample.shape)
print("Positive rate:", (train_sample["crime_count"] > 0).mean())

2020 sample: 250513 positive: 11042 zero: 239471
2021 sample: 249829 positive: 11904 zero: 237925
2022 sample: 249829 positive: 13834 zero: 235995
2023 sample: 249829 positive: 14336 zero: 235493

Final shape: (1000000, 16)
Positive rate: 0.051116


In [15]:
print("Memory usage:", train_sample.memory_usage(deep=True).sum() / 1024**2, "MB")
print("\nTarget distribution:")
print(train_sample["crime_count"].value_counts().sort_index())

Memory usage: 52.470290184020996 MB

Target distribution:
crime_count
0     948884
1      45320
2       4997
3        662
4        108
5         16
6          7
7          2
8          1
9          2
11         1
Name: count, dtype: int64


In [12]:
categorical_features = [
    "grid_id",
    "time_period"
]

print(train_sample[categorical_features].dtypes)

grid_id        category
time_period    category
dtype: object


In [13]:
X_train = train_sample.drop(columns=["crime_count", "cmplnt_fr_dt"])
y_train = train_sample["crime_count"]

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

val_2024 = pd.read_parquet(yearly_files[2024])

X_val = val_2024.drop(columns=["crime_count", "cmplnt_fr_dt"])
y_val = val_2024["crime_count"]

print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)
print("Positive rate:", (y_val > 0).mean())

X_train shape: (1000000, 14)
y_train shape: (1000000,)
X_val shape: (8309664, 14)
y_val shape: (8309664,)
Positive rate: 0.05864292467180382


In [18]:
import lightgbm as lgb

model_baseline = lgb.LGBMRegressor(
    objective="poisson",
    n_estimators=2000,
    learning_rate=0.01,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1,
    max_bin=1024
)

In [19]:
model_baseline.fit(
    X_train,
    y_train,
    categorical_feature=categorical_features,
    eval_set=[(X_val, y_val)],
    eval_metric="poisson",
    callbacks=[
        lgb.early_stopping(30),
        lgb.log_evaluation(50)
    ]
)

C:\Users\sri16\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015901 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5859
[LightGBM] [Info] Number of data points in the train set: 1000000, number of used features: 14
[LightGBM] [Info] Start training from score -2.848952
Training until validation scores don't improve for 30 rounds
[50]	valid_0's poisson: 0.223872
[100]	valid_0's poisson: 0.213407
[150]	valid_0's poisson: 0.207821
[200]	valid_0's poisson: 0.204527
[250]	valid_0's poisson: 0.202401
[300]	valid_0's poisson: 0.200964
[350]	valid_0's poisson: 0.2
[400]	valid_0's poisson: 0.19929
[450]	valid_0's poisson: 0.198781
[500]	valid_0's poisson: 0.19839
[550]	valid_0's poisson: 0.198106
[600]	valid_0's poisson: 0.197892
[650]	valid_0's poisson: 0.197756
[700]	valid_0's poisson: 0.197679
[750]	valid_0's poisson: 0.197621
[800]	valid_0's poisson: 0.1

,boosting_type,'gbdt'
,num_leaves,63
,max_depth,-1
,learning_rate,0.01
,n_estimators,2000
,subsample_for_bin,200000
,objective,'poisson'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [20]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

y_pred = model_baseline.predict(X_val)

mae = mean_absolute_error(y_val, y_pred)
rmse = mean_squared_error(y_val, y_pred) ** 0.5

sample_weights_val = np.where(
    y_val > 0,
    5.0,
    1.0
)

weighted_mae = np.average(
    np.abs(y_val - y_pred),
    weights=sample_weights_val
)

weighted_rmse = np.sqrt(
    np.average(
        (y_val - y_pred) ** 2,
        weights=sample_weights_val
    )
)

print("MAE:", mae)
print("RMSE:", rmse)
print("Weighted MAE:", weighted_mae)
print("Weighted RMSE:", weighted_rmse)

MAE: 0.11879425373320801
RMSE: 0.2746577835203316
Weighted MAE: 0.2714834550731007
Weighted RMSE: 0.5106887047941738


In [22]:
import lightgbm as lgb

model_tweedie = lgb.LGBMRegressor(
    objective="tweedie",
    tweedie_variance_power=1.5,
    n_estimators=2000,
    learning_rate=0.01,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1,
    max_bin=1024
)

In [23]:
model_tweedie.fit(
    X_train,
    y_train,
    categorical_feature=categorical_features,
    eval_set=[(X_val, y_val)],
    eval_metric="tweedie",
    callbacks=[
        lgb.early_stopping(30),
        lgb.log_evaluation(50)
    ]
)

C:\Users\sri16\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.039261 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5859
[LightGBM] [Info] Number of data points in the train set: 1000000, number of used features: 14
[LightGBM] [Info] Start training from score -2.848952
Training until validation scores don't improve for 30 rounds
[50]	valid_0's tweedie: 0.925856
[100]	valid_0's tweedie: 0.869291
[150]	valid_0's tweedie: 0.843916
[200]	valid_0's tweedie: 0.833037
[250]	valid_0's tweedie: 0.829028
[300]	valid_0's tweedie: 0.828827
Early stopping, best iteration is:
[277]	valid_0's tweedie: 0.828365


,boosting_type,'gbdt'
,num_leaves,63
,max_depth,-1
,learning_rate,0.01
,n_estimators,2000
,subsample_for_bin,200000
,objective,'tweedie'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [24]:
y_pred_tweedie = model_tweedie.predict(X_val)

In [25]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

mae_tweedie = mean_absolute_error(y_val, y_pred_tweedie)
rmse_tweedie = mean_squared_error(y_val, y_pred_tweedie) ** 0.5

sample_weights_val = np.where(
    y_val > 0,
    5.0,
    1.0
)

weighted_mae_tweedie = np.average(
    np.abs(y_val - y_pred_tweedie),
    weights=sample_weights_val
)

weighted_rmse_tweedie = np.sqrt(
    np.average(
        (y_val - y_pred_tweedie) ** 2,
        weights=sample_weights_val
    )
)

print("Tweedie MAE:", mae_tweedie)
print("Tweedie RMSE:", rmse_tweedie)
print("Tweedie Weighted MAE:", weighted_mae_tweedie)
print("Tweedie Weighted RMSE:", weighted_rmse_tweedie)

Tweedie MAE: 0.11763489631644801
Tweedie RMSE: 0.2761254682211082
Tweedie Weighted MAE: 0.27679748562398604
Tweedie Weighted RMSE: 0.5225552502362497


In [11]:
calibration_df = pd.DataFrame({
    "prediction": y_pred,
    "actual": y_val
})

bins = [0, 0.1, 0.2, 0.5, 1.0, float("inf")]
labels = ["0.00–0.10", "0.10–0.20", "0.20–0.50", "0.50–1.00", "1.00+"]

calibration_df["prediction_range"] = pd.cut(
    calibration_df["prediction"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

calibration = calibration_df.groupby(
    "prediction_range",
    observed=False
).agg(
    rows=("actual", "size"),
    mean_prediction=("prediction", "mean"),
    mean_actual=("actual", "mean")
).reset_index()

calibration

NameError: name 'y_pred' is not defined

In [12]:
calibration_df_tweedie = pd.DataFrame({
    "prediction": y_pred_tweedie,
    "actual": y_val
})

bins = [0, 0.1, 0.2, 0.5, 1.0, float("inf")]
labels = ["0.00–0.10", "0.10–0.20", "0.20–0.50", "0.50–1.00", "1.00+"]

calibration_df_tweedie["prediction_range"] = pd.cut(
    calibration_df_tweedie["prediction"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

calibration_tweedie = calibration_df_tweedie.groupby(
    "prediction_range",
    observed=False
).agg(
    rows=("actual", "size"),
    mean_prediction=("prediction", "mean"),
    mean_actual=("actual", "mean")
).reset_index()

calibration_tweedie

NameError: name 'y_pred_tweedie' is not defined

In [27]:
print("Mean actual:", y_val.mean())
print("Mean Tweedie prediction:", y_pred_tweedie.mean())

Mean actual: 0.06802838237502744
Mean Tweedie prediction: 0.07361755584012218


In [30]:
sample_weights = np.where(
    y_train > 0,
    5.0,
    1.0
)

model_weighted = lgb.LGBMRegressor(
    objective="poisson",
    n_estimators=2000,
    learning_rate=0.01,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1
)

model_weighted.fit(
    X_train,
    y_train,
    sample_weight=sample_weights,
    categorical_feature=categorical_features,
    eval_set=[(X_val, y_val)],
    eval_metric="poisson",
    callbacks=[
        lgb.early_stopping(30),
        lgb.log_evaluation(50)
    ]
)

C:\Users\sri16\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.013555 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2368
[LightGBM] [Info] Number of data points in the train set: 1000000, number of used features: 14
[LightGBM] [Info] Start training from score -1.425548
Training until validation scores don't improve for 30 rounds
[50]	valid_0's poisson: 0.322193
[100]	valid_0's poisson: 0.313929
[150]	valid_0's poisson: 0.308777
[200]	valid_0's poisson: 0.305166
[250]	valid_0's poisson: 0.30209
[300]	valid_0's poisson: 0.299698
[350]	valid_0's poisson: 0.297557
[400]	valid_0's poisson: 0.295737
[450]	valid_0's poisson: 0.29

,boosting_type,'gbdt'
,num_leaves,63
,max_depth,-1
,learning_rate,0.01
,n_estimators,2000
,subsample_for_bin,200000
,objective,'poisson'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [32]:
print("NaNs in X_train:", X_train.isna().sum().sum())
print("NaNs in X_val:", X_val.isna().sum().sum())
print("NaNs in y_train:", y_train.isna().sum())
print("NaNs in y_val:", y_val.isna().sum())

NaNs in X_train: 0
NaNs in X_val: 0
NaNs in y_train: 0
NaNs in y_val: 0


In [33]:
y_pred_weighted = model_weighted.predict(X_val)

mae_weighted_model = mean_absolute_error(y_val, y_pred_weighted)
rmse_weighted_model = mean_squared_error(y_val, y_pred_weighted) ** 0.5

weighted_mae_weighted_model = np.average(
    np.abs(y_val - y_pred_weighted),
    weights=sample_weights_val
)

weighted_rmse_weighted_model = np.sqrt(
    np.average(
        (y_val - y_pred_weighted) ** 2,
        weights=sample_weights_val
    )
)

print("MAE:", mae_weighted_model)
print("RMSE:", rmse_weighted_model)
print("Weighted MAE:", weighted_mae_weighted_model)
print("Weighted RMSE:", weighted_rmse_weighted_model)

MAE: 0.23115592991537964
RMSE: 0.3575969283440172
Weighted MAE: 0.30206445926426595
Weighted RMSE: 0.46368851847865833


In [34]:
import psutil

memory = psutil.virtual_memory()

print(f"Total RAM: {memory.total / (1024**3):.2f} GB")
print(f"Used RAM: {memory.used / (1024**3):.2f} GB")
print(f"Available RAM: {memory.available / (1024**3):.2f} GB")
print(f"RAM Usage: {memory.percent}%")

Total RAM: 15.64 GB
Used RAM: 12.18 GB
Available RAM: 3.46 GB
RAM Usage: 77.9%


In [35]:
comparison = pd.DataFrame({
    "actual": y_val,
    "unweighted": y_pred,
    "weighted": y_pred_weighted
})

summary = comparison.groupby("actual").agg(
    rows=("actual", "size"),
    unweighted_mean=("unweighted", "mean"),
    weighted_mean=("weighted", "mean")
)

print(summary.head(12))

           rows  unweighted_mean  weighted_mean
actual                                         
0       7822361         0.068956       0.207907
1        422677         0.218784       0.541798
2         54134         0.341262       0.727835
3          8443         0.456706       0.865850
4          1571         0.560701       0.968420
5           340         0.683564       1.077184
6            86         0.852829       1.244594
7            19         0.729084       1.130506
8            12         0.813126       1.216927
9             5         0.741518       1.171827
10            4         0.567554       0.881108
11            1         1.306094       2.012782


In [36]:
sample_weights_2x = np.where(
    y_train > 0,
    2.0,
    1.0
)

model_weighted_2x = lgb.LGBMRegressor(
    objective="poisson",
    n_estimators=2000,
    learning_rate=0.01,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1
)

model_weighted_2x.fit(
    X_train,
    y_train,
    sample_weight=sample_weights_2x,
    categorical_feature=categorical_features,
    eval_set=[(X_val, y_val)],
    eval_metric="poisson",
    callbacks=[
        lgb.early_stopping(30),
        lgb.log_evaluation(100)
    ]
)

C:\Users\sri16\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015225 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2368
[LightGBM] [Info] Number of data points in the train set: 1000000, number of used features: 14
[LightGBM] [Info] Start training from score -2.205657
Training until validation scores don't improve for 30 rounds
[100]	valid_0's poisson: 0.23145
[200]	valid_0's poisson: 0.224604
[300]	valid_0's poisson: 0.22166
[400]	valid_0's poisson: 0.220082
[500]	valid_0's poisson: 0.219035
[600]	valid_0's poisson: 0.218306
[700]	valid_0's poisson: 0.217792
[800]	valid_0's poisson: 0.217342
[900]	valid_0's poisson: 0.21

,boosting_type,'gbdt'
,num_leaves,63
,max_depth,-1
,learning_rate,0.01
,n_estimators,2000
,subsample_for_bin,200000
,objective,'poisson'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [37]:
y_pred_2x = model_weighted_2x.predict(X_val)

mae_2x = mean_absolute_error(y_val, y_pred_2x)
rmse_2x = mean_squared_error(y_val, y_pred_2x) ** 0.5

weighted_mae_2x = np.average(
    np.abs(y_val - y_pred_2x),
    weights=sample_weights_val
)

weighted_rmse_2x = np.sqrt(
    np.average(
        (y_val - y_pred_2x) ** 2,
        weights=sample_weights_val
    )
)

print("MAE:", mae_2x)
print("RMSE:", rmse_2x)
print("Weighted MAE:", weighted_mae_2x)
print("Weighted RMSE:", weighted_rmse_2x)

MAE: 0.15469080373204214
RMSE: 0.2904679824967441
Weighted MAE: 0.2767997352521478
Weighted RMSE: 0.4786832495555617


In [3]:
import os
import psutil

memory = psutil.virtual_memory()

print("Python PID:", os.getpid())
print("CPU cores:", os.cpu_count())
print("Total RAM:", round(memory.total / 1024**3, 2), "GB")
print("Used RAM:", round(memory.used / 1024**3, 2), "GB")
print("Available RAM:", round(memory.available / 1024**3, 2), "GB")

Python PID: 24580
CPU cores: 16
Total RAM: 15.64 GB
Used RAM: 12.87 GB
Available RAM: 2.77 GB


In [14]:
import lightgbm as lgb

model_poisson_127 = lgb.LGBMRegressor(
    objective="poisson",
    n_estimators=2000,
    learning_rate=0.01,
    num_leaves=127,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1,
    max_bin=1024
)

In [15]:
model_poisson_127.fit(
    X_train,
    y_train,
    categorical_feature=categorical_features,
    eval_set=[(X_val, y_val)],
    eval_metric="poisson",
    callbacks=[
        lgb.early_stopping(30),
        lgb.log_evaluation(50)
    ]
)

C:\Users\sri16\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.016361 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5859
[LightGBM] [Info] Number of data points in the train set: 1000000, number of used features: 14
[LightGBM] [Info] Start training from score -2.848952
Training until validation scores don't improve for 30 rounds
[50]	valid_0's poisson: 0.223438
[100]	valid_0's poisson: 0.212999
[150]	valid_0's poisson: 0.207432
[200]	valid_0's poisson: 0.204173
[250]	valid_0's poisson: 0.202092
[300]	valid_0's poisson: 0.200703
[350]	valid_0's poisson: 0.199771
[400]	valid_0's poisson: 0.199138
[450]	valid_0's poisson: 0.198684
[500]	valid_0's poisson: 0.19836
[550]	valid_0's poisson: 0.198085
[600]	valid_0's poisson: 0.197918
[650]	valid_0's poisson: 0.197798
[700]	valid_0's poisson: 0.197734
[750]	valid_0's poisson: 0.197724
[800]	valid_0's poisso

,boosting_type,'gbdt'
,num_leaves,127
,max_depth,-1
,learning_rate,0.01
,n_estimators,2000
,subsample_for_bin,200000
,objective,'poisson'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [16]:
import numpy as np

chunk_size = 500_000
predictions = []

for start in range(0, len(X_val), chunk_size):
    end = min(start + chunk_size, len(X_val))
    
    predictions.append(
        model_poisson_127.predict(X_val.iloc[start:end])
    )

y_pred_127 = np.concatenate(predictions)

print("Predictions:", len(y_pred_127))

Predictions: 8309664


In [17]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae_127 = mean_absolute_error(y_val, y_pred_127)

rmse_127 = mean_squared_error(
    y_val,
    y_pred_127
) ** 0.5

sample_weights_val = np.where(
    y_val > 0,
    5.0,
    1.0
)

weighted_mae_127 = np.average(
    np.abs(y_val - y_pred_127),
    weights=sample_weights_val
)

weighted_rmse_127 = np.sqrt(
    np.average(
        (y_val - y_pred_127) ** 2,
        weights=sample_weights_val
    )
)

print("Poisson 127 Leaves")
print("------------------")
print("MAE:", mae_127)
print("RMSE:", rmse_127)
print("Weighted MAE:", weighted_mae_127)
print("Weighted RMSE:", weighted_rmse_127)

Poisson 127 Leaves
------------------
MAE: 0.11783950168915086
RMSE: 0.2748599799613795
Weighted MAE: 0.2705399985041401
Weighted RMSE: 0.5107081168545679


In [18]:
import sys

print(sys.executable)

import pandas
import pyarrow
import lightgbm

print("Environment OK")

C:\Users\sri16\AppData\Local\Programs\Python\Python313\python.exe
Environment OK
